# LeetCode #1098: Sort the Matrix Diagonally

https://leetcode.com/problems/sort-the-matrix-diagonally/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m \times n \times \min(m,n))$ | $O(m \times n)$ |
| **Optimal: Diagonal Key Grouping ★** | $O(m \times n \times \log \min(m,n))$ | $O(m \times n)$ |

---

## Understanding the Methods

### Brute Force
For every starting cell of a diagonal, extract elements one by one, sort with a generic sort, and write back — repeated inner loops add constant overhead but the same $O(k \log k)$ sorting per diagonal.

### Optimal: Diagonal Key Grouping ★
Every cell `(i, j)` belongs to the diagonal identified by the key `i - j`. Group cells by key into lists, sort each list, and overwrite the diagonal in place. One pass to collect, $O(k \log k)$ per diagonal of length $k$, one pass to scatter back.

**Why this is better than Brute Force:** Same asymptotic complexity, but grouping via a dictionary avoids re-traversing diagonal start positions and makes the code cleaner and cache-friendlier.

**Constraints:**
* $m == \text{mat.length}$, $n == \text{mat}[i]\text{.length}$
* $1 \leq m, n \leq 100$
* $1 \leq \text{mat}[i][j] \leq 100$


## Solutions

### C#

In [ ]:
public class Solution {
    public int[][] DiagonalSort(int[][] mat) {
        int m = mat.Length, n = mat[0].Length;
        var diags = new Dictionary<int, List<int>>();

        // Group each cell's value under its diagonal key (row - col)
        for (int i = 0; i < m; i++)
            for (int j = 0; j < n; j++) {
                int key = i - j;
                if (!diags.ContainsKey(key)) diags[key] = new List<int>();
                diags[key].Add(mat[i][j]);
            }

        // Sort every diagonal so elements flow top-left to bottom-right in order
        foreach (var kv in diags) kv.Value.Sort();

        // Scatter sorted values back, consuming each diagonal list front-to-back
        var pos = new Dictionary<int, int>();
        for (int i = 0; i < m; i++)
            for (int j = 0; j < n; j++) {
                int key = i - j;
                if (!pos.ContainsKey(key)) pos[key] = 0;
                mat[i][j] = diags[key][pos[key]++];
            }

        return mat;
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        diags: dict[int, list[int]] = defaultdict(list)

        # Collect each cell into the bucket for its diagonal (row - col is constant)
        for i in range(m):
            for j in range(n):
                diags[i - j].append(mat[i][j])

        # Sort each diagonal so ascending order runs from top-left to bottom-right
        for key in diags:
            diags[key].sort()

        # Write sorted values back in the same top-left-first traversal order
        idx: dict[int, int] = defaultdict(int)
        for i in range(m):
            for j in range(n):
                key = i - j
                mat[i][j] = diags[key][idx[key]]
                idx[key] += 1

        return mat

### Go

In [ ]:
import "sort"

func diagonalSort(mat [][]int) [][]int {
    m, n := len(mat), len(mat[0])
    diags := map[int][]int{}

    // Collect values by diagonal key (row - col is invariant along a diagonal)
    for i := 0; i < m; i++ {
        for j := 0; j < n; j++ {
            key := i - j
            diags[key] = append(diags[key], mat[i][j])
        }
    }

    // Sort each diagonal group ascending
    for key := range diags {
        sort.Ints(diags[key])
    }

    // Scatter sorted values back in the same traversal order
    pos := map[int]int{}
    for i := 0; i < m; i++ {
        for j := 0; j < n; j++ {
            key := i - j
            mat[i][j] = diags[key][pos[key]]
            pos[key]++
        }
    }
    return mat
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn diagonal_sort(mut mat: Vec<Vec<i32>>) -> Vec<Vec<i32>> {
        let m = mat.len();
        let n = mat[0].len();
        let mut diags: HashMap<i32, Vec<i32>> = HashMap::new();

        // Group each cell's value by its diagonal key (row - col)
        for i in 0..m {
            for j in 0..n {
                let key = i as i32 - j as i32;
                diags.entry(key).or_default().push(mat[i][j]);
            }
        }

        // Sort every diagonal so smallest element sits at the top-left end
        for vals in diags.values_mut() {
            vals.sort_unstable();
        }

        // Write sorted values back; same traversal order guarantees correct placement
        let mut pos: HashMap<i32, usize> = HashMap::new();
        for i in 0..m {
            for j in 0..n {
                let key = i as i32 - j as i32;
                let p = pos.entry(key).or_insert(0);
                mat[i][j] = diags[&key][*p];
                *p += 1;
            }
        }
        mat
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `mat = [[3,3,1,1],[2,2,1,2],[1,1,1,2]]`
Diagonal $i-j=0$ has values `[3,2,1]` → sorted `[1,2,3]`. Each of the $m+n-1 = 6$ diagonals is sorted independently and written back, yielding `[[1,1,1,1],[1,2,2,2],[1,2,3,3]]`.

### 2. Slightly Complex
**Input:** `mat = [[11,25,5],[5,28,13],[8,4,30]]` ($3 \times 3$)
Seven diagonals (lengths 1, 2, 3, 2, 1 for corner diagonals). Main diagonal `[11,28,30]` sorts to itself; off-diagonals each sort independently. No element crosses diagonal boundaries.

### 3. Edge Case: Time Factor
**Input:** $m = n = 100$ (maximum size)
The main diagonal has length 100: sorting it alone costs $O(100 \log 100) \approx 665$ comparisons. Total across all diagonals: $O(m \times n \times \log \min(m,n)) \approx 66{,}500$ comparisons — confirming the dominant term.

### 4. Edge Case: Space Factor
**Input:** $m = 1$, $n = 100$ (single row)
Every cell is its own length-1 diagonal (key = $0 - j$ for $j = 0 \ldots 99$). No sort needed; 100 length-1 lists are allocated totalling $O(n)$ auxiliary space — the worst-case $O(m \times n)$ bound simplifies to $O(n)$ here.

### 5. Almost-Impossible but Plausible
**Input:** `mat = [[100,100],[100,100]]` (all same value)
All diagonals are already sorted (trivially). The algorithm runs all passes, sorts no-op lists, and returns the matrix unchanged. Confirms stability when duplicates dominate.
